<a href="https://colab.research.google.com/github/erickvaldezsallagos/SISTEMAS-FINANCIEROS-Y-COMERCIALES/blob/main/TareaB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SISTEMAS FINANCIEROS Y COMERCIALES** \
**EQUIPO 3**\
TAREA B: Realizar un programa consultando una base de datos. Calcule y grafique VWMA y MACD de un menú de acciones y de un menú de tamaños de ventana

In [5]:

#  Instalación e importación de librerías


import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Definición de funciones de cálculo

def calcular_vwma(df, ventana):
    """
    Calcula la Media Móvil Ponderada por Volumen (VWMA):
    VWMA = Suma(Precio * Volumen) / Suma(Volumen) para el período dado.
    """
    precio_volumen = df['Close'] * df['Volume']
    suma_pv = precio_volumen.rolling(window=ventana).sum()
    suma_vol = df['Volume'].rolling(window=ventana).sum()
    return suma_pv / suma_vol

def calcular_macd(df, span_rapido=12, span_lento=26, span_senal=9):
    """
    Calcula el MACD y su línea de señal basados en EMA:
    MACD = EMA(12) - EMA(26)
    Señal = EMA(9) del MACD
    Histograma = MACD - Señal
    """
    ema_rapida = df['Close'].ewm(span=span_rapido, adjust=False).mean()
    ema_lenta = df['Close'].ewm(span=span_lento, adjust=False).mean()
    macd_line = ema_rapida - ema_lenta
    signal_line = macd_line.ewm(span=span_senal, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

# Creación de los Menús Interactivos

# Menú para seleccionar el ticker de la acción
ticker_widget = widgets.Dropdown(
    options=['AAPL', 'MSFT', 'TSLA', 'GOOGL', 'AMZN', 'NVDA'],
    value='AAPL',
    description='Acción:',
    style={'description_width': 'initial'}
)

# Menú para seleccionar la ventana de cálculo (VWMA / Medias)
ventana_widget = widgets.Dropdown(
    options=[10, 20, 50, 100, 200],
    value=20,
    description='Ventana VWMA:',
    style={'description_width': 'initial'}
)

# Botón para actualizar la gráfica
boton_actualizar = widgets.Button(
    description='Generar Gráficas',
    button_style='success',
    icon='line-chart'
)

# Salida donde se mostrarán las gráficas
out = widgets.Output()

def on_button_clicked(b):
    with out:
        clear_output(wait=True)
        ticker = ticker_widget.value
        ventana = ventana_widget.value

        print(f"Descargando datos históricos para {ticker}...")
        # Descargamos datos históricos de los últimos 6 meses
        df = yf.download(ticker, period="6mo", interval="1d", progress=False)

        if df.empty:
            print("No se pudieron obtener datos para el ticker seleccionado.")
            return

        # Limpieza por si yfinance devuelve multi-index en las columnas
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # Cálculo de indicadores
        df['VWMA'] = calcular_vwma(df, ventana)
        df['MACD'], df['Signal'], df['Histogram'] = calcular_macd(df)

        # Visualización
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

        # Gráfica 1: Precio de Cierre y VWMA
        ax1.plot(df.index, df['Close'], label=f'Precio Cierre ({ticker})', color='black', alpha=0.6)
        ax1.plot(df.index, df['VWMA'], label=f'VWMA ({ventana} periodos)', color='blue', linewidth=2)
        ax1.set_title(f'Análisis Técnico: VWMA y MACD para {ticker}', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Precio (USD)')
        ax1.legend(loc='upper left')
        ax1.grid(True, linestyle='--', alpha=0.5)

        # Gráfica 2: MACD e Histograma
        ax2.plot(df.index, df['MACD'], label='MACD Line (12, 26)', color='purple', linewidth=1.5)
        ax2.plot(df.index, df['Signal'], label='Signal Line (9)', color='orange', linewidth=1.5)

        # Histograma con colores dinámicos (verde si sube, rojo si baja)
        colores = np.where(df['Histogram'] >= 0, 'green', 'red')
        ax2.bar(df.index, df['Histogram'], label='Histograma', color=colores, alpha=0.5, width=1.0)

        ax2.set_ylabel('MACD')
        ax2.set_xlabel('Fecha')
        ax2.legend(loc='upper left')
        ax2.grid(True, linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.show()

# Vincular el evento del botón con la función
boton_actualizar.on_click(on_button_clicked)

# Mostrar la interfaz de usuario en Colab
display(widgets.VBox([ticker_widget, ventana_widget, boton_actualizar, out]))